# 📊 Caderno 3: Engenharia Quântica e Validação (Issue #6)

Este caderno valida as **10 Regras de Ouro Quants** implementadas, permitindo analisar Momentum, OBV, Volatilidade em múltiplas janelas e Médias Móveis.

## 1. Configuração e Carga de Dados

In [3]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from ipywidgets import interact, interactive_output, HBox, VBox
import warnings

warnings.filterwarnings('ignore')

# Carrega o Dataset Analítico Quantitativo
caminho_dados = '../data/processed/01_market_data_processed.csv'
df_processado = pd.read_csv(caminho_dados, sep=';', decimal=',')
df_processado['date'] = pd.to_datetime(df_processado['date'])

tickers_disponiveis = sorted(df_processado['ticker'].unique())
print(f"✅ Base Quantitativa Carregada! {len(tickers_disponiveis)} ativos encontrados.")

✅ Base Quantitativa Carregada! 467 ativos encontrados.


## 2. Dashboard Quantitativo Interativo

In [4]:
# Widgets
drop_ticker = widgets.Dropdown(options=tickers_disponiveis, value='PETR4.SA', description='Ativo:')
check_sma = widgets.SelectMultiple(options=['sma_21', 'sma_50', 'sma_200'], value=['sma_21', 'sma_200'], description='Médias:')

def dashboard_quant(ticker, medias):
    df_at = df_processado[df_processado['ticker'] == ticker].sort_values('date').copy()
    
    # Painel de Subplots
    fig = make_subplots(rows=4, cols=1, 
                        shared_xaxes=True, 
                        vertical_spacing=0.05,
                        subplot_titles=(f'Preço e Tendência: {ticker}', 'Momentum (1m vs 12m)', 'Risco: Volatilidade (21d vs 252d)', 'Fluxo: OBV'),
                        row_heights=[0.4, 0.2, 0.2, 0.2])

    # 1. Preço e Médias
    fig.add_trace(go.Scatter(x=df_at['date'], y=df_at['close'], name='Preço', line=dict(color='white')), row=1, col=1)
    colors_sma = {'sma_21': 'cyan', 'sma_50': 'orange', 'sma_200': 'yellow'}
    for m in medias:
        fig.add_trace(go.Scatter(x=df_at['date'], y=df_at[m], name=m, line=dict(color=colors_sma[m], dash='dot')), row=1, col=1)

    # 2. Momentum
    fig.add_trace(go.Scatter(x=df_at['date'], y=df_at['momentum_1m'], name='Momentum 1M', fill='tozeroy'), row=2, col=1)
    fig.add_trace(go.Scatter(x=df_at['date'], y=df_at['momentum_12m'], name='Momentum 12M'), row=2, col=1)

    # 3. Volatilidade
    fig.add_trace(go.Scatter(x=df_at['date'], y=df_at['volatilidade_21d'], name='Vol 21d (Curto)'), row=3, col=1)
    fig.add_trace(go.Scatter(x=df_at['date'], y=df_at['volatilidade_252d'], name='Vol 252d (Longo)'), row=3, col=1)

    # 4. OBV
    fig.add_trace(go.Scatter(x=df_at['date'], y=df_at['obv'], name='OBV', line=dict(color='lime')), row=4, col=1)

    fig.update_layout(height=900, template='plotly_dark', showlegend=True, title_text=f"Análise Quantitativa Profissional: {ticker}")
    fig.show()
    
    # Info Box
    last_ret = df_at['retorno_acumulado'].iloc[-1]
    last_obv = df_at['obv'].iloc[-1]
    trend = "Alta (Golden Cross!)" if df_at['tendencia_alta_50_200'].iloc[-1] == 1 else "Baixa (Death Cross!)"
    
    print(f"📊 RESUMO QUANT ({ticker}):")
    print(f"   - Retorno Histórico: {last_ret:.2%}")
    print(f"   - Tendência Longo Prazo: {trend}")
    print(f"   - OBV Final: {last_obv:,.0f}")

ui = HBox([drop_ticker, check_sma])
out = interactive_output(dashboard_quant, {'ticker': drop_ticker, 'medias': check_sma})
display(VBox([ui, out]))